In [ ]:
import sys
from contextlib import redirect_stdout 
import io
import pandas as pd

sys.path.append('../')
sys.path.append('../../src')

df = pd.read_parquet('../ssi_all_features.parquet')
df.shape

In [ ]:
df_pivoted = df.pivot(index='birthid', columns='feature_name', values='feature_value').reset_index()
df_pivoted.columns.name = ''
df_pivoted

In [ ]:
from preprocess import preprocess

buf = io.StringIO()
with redirect_stdout(buf):
    df_pivoted_preprocessed = preprocess(df_pivoted)

In [ ]:
df_pivoted_preprocessed.shape

In [ ]:
from feature_filter import feature_filter

buf = io.StringIO()
with redirect_stdout(buf):
    df_pivoted_filtered = feature_filter(df_pivoted_preprocessed, skip_medadmin_ingredient=True)

del df_pivoted_preprocessed

In [ ]:
df_pivoted_filtered.shape

In [ ]:
pcornet_features = [x for x in df_pivoted_filtered.columns \
                    if x != 'target_variable' and x != 'birthid' and x != 'mother_patid']
all_features = []
for x in pcornet_features:
    if 'medadmin_ingredient' in x:
        if 'has_medadmin_ingredient' in x or 'earliest_minute' in x:
            all_features.append(x)
    else:
        all_features.append(x)

print(len(all_features))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from model_training import loop_over_selector
from feature_selectors import CachedTopKSelector

selector_basemodel = RandomForestClassifier(class_weight='balanced', random_state=42, min_samples_split=100)

pipeline = Pipeline([
    ('selector', CachedTopKSelector(selector_basemodel, method='shap')),
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=42))
])

param_grid = {
    'selector__k': [10, 20, 30, 50, 100, 200, 300, 500, 1000, 3000, 5000],
    'clf__min_samples_split': [5, 20, 50, 70, 100, 200, 500, 700, 1000]
}

X = df_pivoted_filtered[all_features]
y = df_pivoted_filtered['target_variable']
mother_patid = df_pivoted_filtered['mother_patid']

metrics = loop_over_selector(X, y, pipeline, param_grid, outer_splits=5, inner_splits=5, groups=mother_patid,
                             scoring='roc_auc', n_jobs=1)

In [ ]:
metrics

In [ ]:
metrics[metrics['number_of_features']==300].describe()

In [ ]:
metrics.to_csv('final_model_and_shap_for_paper_selector_k.csv', index=False)

In [ ]:
from matplotlib import pyplot as plt

fig, axe = plt.subplots(1, 1, figsize=(8, 5))
tmp3 = metrics.groupby('selector__k')[['number_of_features', 'roc_auc']].agg(['mean', 'std'])
ks = tmp3['number_of_features']['mean']
axe.errorbar(ks, tmp3['roc_auc']['mean'], tmp3['roc_auc']['std'], fmt='o-', capsize=5, linewidth=1.0)
axe.set_xlabel('Number of Features')
axe.set_ylabel('ROC_AUC')
axe.set_ylim([0.60, 0.77])

In [ ]:
from collections import defaultdict

features_at_300 = metrics.loc[metrics['number_of_features']==300, 'selected_features']
counts = defaultdict(lambda: 0)
for feature_list in features_at_300:
    for feat in feature_list:
        counts[feat] += 1
tmp = pd.DataFrame(counts.items(), columns=['features', 'mentioned_times']).sort_values(by='mentioned_times', ascending=False)
tmp.head(30)

In [ ]:
features_selected = tmp.loc[tmp['mentioned_times']>=3, 'features'].tolist()
features_selected, len(features_selected)

In [ ]:
from model_training import nested_cv_evaluation_feature_selection

X = df_pivoted_filtered[features_selected]
print(X.shape)
y = df_pivoted_filtered['target_variable']
mother_patid = df_pivoted_filtered['mother_patid']
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
param_grid = {'min_samples_split': [20, 50, 70, 100, 200, 500]}

metrics_baseline = nested_cv_evaluation_feature_selection(X, y, clf, param_grid, groups=mother_patid,
                                                          outer_splits=5, inner_splits=5, random_state=range(10),
                                                          return_for_curves=True)

In [ ]:
metrics_baseline

In [ ]:
from model_training import nested_cv_evaluation_feature_selection

X = df_pivoted_filtered[features_selected]
print(X.shape)
y = df_pivoted_filtered['target_variable']
mother_patid = df_pivoted_filtered['mother_patid']
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
param_grid = {'min_samples_split': [20, 50, 70, 100, 200, 500]}

metrics_baseline = nested_cv_evaluation_feature_selection(X, y, clf, param_grid, groups=mother_patid,
                                                          outer_splits=5, inner_splits=5, random_state=range(10),
                                                          return_for_curves=True)

In [ ]:
y.mean()

In [ ]:
from my_scorers import boostrap_ci_agg

metrics_baseline[['roc_auc']].apply(boostrap_ci_agg)

In [ ]:
metrics_baseline[['ap']].apply(boostrap_ci_agg), metrics_baseline[['sens_80']].apply(boostrap_ci_agg)

In [ ]:
metrics_baseline.describe()

In [ ]:
metrics_baseline.to_csv('final_model_and_shap_for_paper_roc_auc.csv', index=False)

In [ ]:
metrics_baseline['best_params'].apply(lambda x: x['min_samples_split']).describe()

In [ ]:
from my_scorers import plot_nested_cv_with_95ci

fig = plot_nested_cv_with_95ci(metrics_baseline)

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

X = df_pivoted_filtered[features_selected]
print(X.shape)
y = df_pivoted_filtered['target_variable']
mother_patid = df_pivoted_filtered['mother_patid']
clf = GridSearchCV(RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'),
                   param_grid={'min_samples_split': [30, 40, 50, 60, 70, 80, 90, 100]},
                   scoring='roc_auc',
                   verbose=3,
                   cv=StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42))
clf.fit(X, y, groups=mother_patid)

In [ ]:
clf.best_params_, clf.best_score_

In [ ]:
pd.DataFrame({'features': features_selected}).to_csv('final_model_and_shap_for_paper_features_selected.csv')

In [ ]:
clf.best_estimator_

In [ ]:
import pickle
import json

metadata = {
    'min_sample_splits': 80,
    "n_estimators": 200,
    'selected_features': features_selected
}

with open("../models/model_metadata.json", "w") as fid:
    json.dump(metadata, fid, indent=2)

with open('../models/best_model.pickle', 'wb') as fid:
    pickle.dump(clf.best_estimator_, fid)

In [ ]:
import shap

def get_shap(clf, X, max_display=30):
    explainer = shap.TreeExplainer(clf, data=X,
                                   model_output='probability', feature_perturbation='interventional')
    shap_values = explainer(X, check_additivity=True)
    shap_values.values = shap_values.values[:,:,1]
    shap.plots.beeswarm(shap_values, max_display=max_display)
    return shap_values

shap_values = get_shap(clf.best_estimator_, X, max_display=50)

### 2. break down the shap

In [ ]:
import numpy as np

df_shap = pd.DataFrame({
    'feature': X.columns,
    'shap': np.mean(np.abs(shap_values.values), axis=0)
}).sort_values(by='shap', ascending=False)

df_shap.shape

In [ ]:
df_shap.to_csv('final_model_and_shap_for_paper_shap.csv', index=False)

In [ ]:
df_shap

In [ ]:
feature_grouping = df.groupby(['description'])['feature_name'].unique().apply(list).to_dict()
list(feature_grouping.keys())

In [ ]:
import seaborn as sns
from matplotlib.ticker import MaxNLocator

todo = {
    'bp': ['all_bp', 'before_delivery_bp', 'after_delivery_bp'],
    'wt': ['wt', 'original_bmi', 'computed_bmi'],
    'lab': ['glucose', 'ebl'],
    'static': ['baseline'],
    'medication': ['before_delivery_medadmin_ingredient', 'after_delivery_medadmin_ingredient'],
    'diagnosis': ['diagnosis'],
    'OR timing': ['or_timings']
}

diagnosis_dict = {
    "O99_213_has_dx": "Obesity complicating pregnancy, third trimester",
    "O99_212_has_dx": "Obesity complicating pregnancy, second trimester",
    "E66_0_has_dx": "Obesity due to excess calories",
    "O09_892_has_dx": "Supervision of other high-risk pregnancies, second trimester",
    "O26_892_has_dx": "Other specified pregnancy-related conditions, second trimester",
    "O99_332_has_dx": "Smoking (tobacco) complicating pregnancy, second trimester",
    "Z13_79_has_dx": "Encounter for other screening for genetic and chromosomal anomalies",
    "O36_8130_has_dx": "Decreased fetal movements, third trimester, unsp",
    "Z98_891_has_dx": "History of uterine scar from previous surgery",
    "O99_824_has_dx": "Streptococcus B carrier state complicating childbirth"
}

med_dict = {
    '161': "Acetaminophen",
    '114202+1901+8591+9863': 'Lactated ringers',
    '1815': "Bupivacaine",
    '26225': "Ondansetron",
    '9863': "Sodium chloride (flush)",
    '6387': "Lidocaine",
    "2582": "Clindamycin",
    "5640": "Ibuprofen"
}

def replace_med_code(feature):
    for code, desc in med_dict.items():
        if feature.startswith(code):
            return feature.replace(code, desc, 1)
    return feature


def wrap_middle(text):
    if len(text) <= 30:
        return text
    parts = text.split()
    mid = len(parts) // 2
    return " ".join(parts[:mid]) + "\n" + " ".join(parts[mid:])


for group, key_list in todo.items():
    features = []
    for key in key_list:
        features.extend(feature_grouping[key])
    df_shap_filtered = df_shap[df_shap['feature'].isin(features)].copy()

    # rename the column so more readable
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('has_medadmin_ingredient', 'Y/N')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('medadmin_ingredient', '')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('INTRAVENOUS', '(IV)')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('ORAL', '(oral)')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('UNKNOWN', '')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].apply(replace_med_code)

    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('diastolic_all', 'diastolic')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('systolic_all', 'systolic')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('bp_all', 'bp')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('or_timings', '')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('datetime', 'time')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('has_cmqcc_category', '_(cmqcc_category)')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].apply(lambda x: diagnosis_dict.get(x, x))

    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('anes_', 'anesthesia_')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('disch_', 'discharge_')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('fac_', 'facility_')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('sched_', 'scheduled_')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('proc_', 'procedure_')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('recov_', 'recovery_')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('comp_', 'complete_')

    df_shap_filtered['feature'] = df_shap_filtered['feature'].str.replace('_', ' ')
    df_shap_filtered['feature'] = df_shap_filtered['feature'].apply(wrap_middle)

    if group == 'diagnosis' or group == 'medication' or group == 'static' or group == 'OR timing':
        n = 10
        plt.figure(figsize=(3, 6))
    else:
        n = 5
        plt.figure(figsize=(3, 3))

    ax = sns.barplot(df_shap_filtered.head(n), x='shap', y='feature')
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    plt.title(group)
    plt.ylabel('')

### 3. remake the shap plot using the same mapping

In [ ]:
df_shap_tmp = pd.DataFrame({'feature': shap_values.feature_names})

# rename the column so more readable
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('has_medadmin_ingredient', 'Y/N')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('medadmin_ingredient', '')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('INTRAVENOUS', '(IV)')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('ORAL', '(oral)')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('UNKNOWN', '')
df_shap_tmp['feature'] = df_shap_tmp['feature'].apply(replace_med_code)

df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('diastolic_all', 'diastolic')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('systolic_all', 'systolic')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('bp_all', 'bp')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('or_timings', '')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('datetime', 'time')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('has_cmqcc_category', '_(cmqcc_category)')
df_shap_tmp['feature'] = df_shap_tmp['feature'].apply(lambda x: diagnosis_dict.get(x, x))

df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('anes_', 'anesthesia_')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('disch_', 'discharge_')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('fac_', 'facility_')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('sched_', 'scheduled_')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('proc_', 'procedure_')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('recov_', 'recovery_')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('comp_', 'complete_')
df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('case_req', 'case_request')

df_shap_tmp['feature'] = df_shap_tmp['feature'].str.replace('_', ' ')

In [ ]:
from copy import deepcopy

shap_values_copy = deepcopy(shap_values)
# shap_values_copy.feature_names = df_shap_tmp['feature'].values.tolist()

top_idx = np.argsort(np.abs(shap_values_copy.values).mean(0))[-40:]

shap_values_copy.data = shap_values_copy.data[:, top_idx]
shap_values_copy.base_values = shap_values_copy.base_values[:]
shap_values_copy.values = shap_values_copy.values[:, top_idx]
shap_values_copy.feature_names = [df_shap_tmp['feature'].values[i] for i in top_idx]

shap.plots.beeswarm(shap_values_copy, max_display=40)

### 4 some more analysis for the best model

In [ ]:
X.shape, len(features_selected), clf.best_estimator_

In [ ]:
metrics_baseline.describe()

### 5. any improvement on sens_80 is i optimize it?

In [ ]:
import sys
del sys.modules['model_training']

from model_training import sense_80, nested_cv_evaluation_feature_selection

X = df_pivoted_filtered[features_selected]
print(X.shape)
y = df_pivoted_filtered['target_variable']
mother_patid = df_pivoted_filtered['mother_patid']
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
param_grid = {'min_samples_split': [20, 50, 70, 100, 200, 500]}

metrics_optimized_sens_80 = nested_cv_evaluation_feature_selection(X, y, clf, param_grid, groups=mother_patid,
                                scoring=sense_80, outer_splits=5, inner_splits=5, random_state=range(10))

In [ ]:
metrics_optimized_sens_80.describe()

Not much difference